# 제3 테스트셋 (BFCL) 생성 (논문에 없는 추가)
BFCL v4 질의에 우리 함수 기준 정답을 붙인 test 셋(설명서 `BFCL_TEST.md`)으로 우리 모델과 공개 모델의 출력을 만든다.
생성·채점 모두 `src/evaluate.py` (BFCL 필드 `score_args`·`args_allowed` 처리 포함), 확률은 `src/probs.py`.

- 런타임: **A100 GPU**. 🔑 Secrets에 `HF_TOKEN`
- test 파일: Drive `MyDrive/octopus_v2/bfcl_test_draft.jsonl` (234개, **사용자 검토 전 초안**). 검토 후 바뀐 줄만 다시 채점하면 된다
- 결과: Drive `MyDrive/octopus_v2/runs/eval_bfcl/{ours,public}/` 에 `preds.jsonl`, `probs.jsonl`

## 준비 (그냥 실행)

In [ ]:
!pip -q install transformers==4.56.2 accelerate
from huggingface_hub import login
from google.colab import userdata, drive

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

import os
REPO = "/content/octopus_v2_reproduce"
if not os.path.exists(REPO):
    !git clone https://github.com/nahcub/octopus_v2_reproduce.git {REPO}
%cd {REPO}
!git pull
!git log --oneline -1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers; print("transformers", transformers.__version__)  # 4.56.2 여야 함

RUN_DIR = "/content/drive/MyDrive/octopus_v2/runs/full"   # 4단계에서 학습한 모델
EVAL = "/content/drive/MyDrive/octopus_v2/runs/eval_bfcl"
TEST = "/content/drive/MyDrive/octopus_v2/bfcl_test_draft.jsonl"
os.makedirs(EVAL, exist_ok=True)
!wc -l {TEST}

---
## 7-1. 우리 모델로 생성 + 확률
**예측**: 우리 모델은 질문형 negative 로 학습해서 질문형 positive(62개)를 거절할 것. 거절 정확도는 높을 것.
> (여기에 예측)

In [ ]:
!python -m src.evaluate generate --model {RUN_DIR} --test-file {TEST} --out {EVAL}/ours/preds.jsonl
!python -m src.probs generate --model {RUN_DIR} --test-file {TEST} --preds {EVAL}/ours/preds.jsonl --out {EVAL}/ours/probs.jsonl

In [ ]:
!python -m src.evaluate score --preds {EVAL}/ours/preds.jsonl --test-file {TEST}

---
## 7-2. 공개 모델 `NexaAIDev/Octopus-v2` 로 생성 + 확률
**예측**: 잡담·상식·계산 negative 에 함수를 부를 것 (우리 test 에서 negative 39%).
> (여기에 예측)

In [ ]:
!python -m src.evaluate generate --model NexaAIDev/Octopus-v2 --test-file {TEST} --out {EVAL}/public/preds.jsonl
!python -m src.probs generate --model NexaAIDev/Octopus-v2 --test-file {TEST} --preds {EVAL}/public/preds.jsonl --out {EVAL}/public/probs.jsonl

In [ ]:
!python -m src.evaluate score --preds {EVAL}/public/preds.jsonl --test-file {TEST}

---
## 끝나면
결과는 Drive `runs/eval_bfcl/` 에 있다. 랩탑에서 Claude 에게 "BFCL 결과 분석해줘" 라고 하면 Drive 에서 받아 AUROC·오답 분석을 한다.